# Quantum Fourier Transform Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Quantum Fourier Transform" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import Qubits


## Problem 1. Implement single-qubit QFT

As we've seen in the discussion of single-qubit QFT, it should transform starting state $\ket{0}$ to $\ket{+}$ and $\ket{1}$ - to $\ket{-}$.
And you already know a gate that will do this: the Hadamard gate!

In [ ]:
def single_qubit_qft(x: Qubits) -> None:
    x.had()

## Problem 2. Rotation gate

To avoid adding the extra global phase, we have to use a gate that does not modify the $\ket{0}$ state and only impacts $\ket{1}$.
The matrix of the required transformation can be then written as follows:

$$\begin{bmatrix} 1 & 0 \\ 0 & e^{2\pi i/2^{k}} \end{bmatrix}$$

The built-in phase gate is exactly the gate we need: 

$$\textrm{Phase}(\theta) = \begin{bmatrix} 1 & 0 \\ 0 & e^{i \theta} \end{bmatrix}$$

We need to use $\theta = \tfrac{2\pi}{2^{k}}$. We can express this argument in a convenient form using an alternative way to specify the rotation angle argument supported by all rotation gates: instead of expressing it as degrees or radians, we can express it as a fraction of $\pi$ using a tuple `(numerator, denominator)` - in our case, `(2, 2 ** k)`.

In [ ]:
def rotation_gate(x: Qubits, k: int) -> None:
    x.phase((2, 2 ** k))

## Problem 3. Binary fraction exponent (with classical input)

The most straightforward solution is to use a single phase gate, similar to what you did in the previous problem. You can calculate the rotation angle for it $2\pi \cdot \overline{0.j_{n-1} j_{n-2} ... j_0}$ by converting the given bit string into an integer $j$ (using little-endian notation) and dividing it by $2^n$.

However, this solution will not be helpful for solving the next problem, in which (spoiler alert!) classical input $j$ will be replaced with quantum one. Let's consider a different approach, based on processing each digit of $j$ separately.

Since you can express the exponent of a sum as a product of exponents ($e^{a+b} = e^a \cdot e^b$), you can implement the required transformation as a sequence of rotation gates from the previous problem with increasing $k$.

Each of the individual rotations will use the bit $j_k$ to decide whether to apply the rotation (you only want to apply the rotation if $j_k$ is true), and the index of that bit $k$ to define the rotation angle.
The gate applied for each $k$ will be:

$$U_k = \begin{cases}
I, j_k=0 \\
\textrm{Phase}(2\pi \cdot \frac{1}{2^{n-k}}), j_k=1
\end{cases} = \textrm{Phase}(2\pi \cdot \frac{j_k}{2^{n-k}})$$

(To see why the denominator of the rotation angle is $2^{n-k}$, recall that $k=0$ corresponds to the least significant bit of $j$ that needs to be divided by $2^n$, $k=1$ corresponds to the second least significant bit that needs to be divided by $2^{n-1}$, and so on.)

As you iterate over $k$ and apply rotations, the resulting state will get closer and closer to the required one, starting with the least significant digits of $j$:

| $k$ | State after step $k$ |
| --- | - |
| $0$ | $x_0 \ket{0} + x_1 \cdot e^{2\pi i \cdot \overline{0.0 ... 0j_0}} \ket{1}$ |
| $1$ | $x_0 \ket{0} + x_1 \cdot e^{2\pi i \cdot \overline{0.0 ... j_1 j_0}} \ket{1}$ |
| ... | ... |
| $n-1$ | $x_0 \ket{0} + x_1 \cdot e^{2\pi i \cdot \overline{0.j_{n-1} ... 0j_0}} \ket{1}$ |

In [ ]:
def binary_fraction_exponent_classical(x: Qubits, j: list[bool]) -> None:
    n = len(j)
    for ind in range(n):
        if j[ind]:
            x.phase((2, 2 ** (n - ind)))

## Problem 4. Binary fraction exponent (with quantum input)

Since $j$ is a quantum register and can be in a superposition of basis states, you cannot just measure the register and then apply the function from the previous problem using measurement results as the second argument. Instead, you have to convert the solution to the previous problem from using $\textrm{Phase}$ gates with classical conditions (applying them only if $j_k = 0$) to using controlled variants of $\textrm{Phase}$ gates, with each of the qubits of the register $j$ as controls.

In [ ]:
def binary_fraction_exponent_quantum(x: Qubits, j: Qubits) -> None:
    n = j.num_qubits
    for ind in range(n):
        x.phase((2, 2 ** (n - ind)), cond=j[ind])

## Problem 5. Binary fraction exponent in-place (with quantum input)

First, let's recall the first task of the kata: a Hadamard gate applied to a single qubit, $H\ket{j_{n-1}}$, will give either
$\frac1{\sqrt2}(\ket{0} + \ket{1})$ or $\frac{1}{\sqrt{2}}(\ket{0} - \ket{1})$ depending on the state of $\ket{j_{n-1}}$. This operation can also be written as

$$H\ket{j_{n-1}} = \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \frac{j_1}{2}}\ket{1} \big)= \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_1}}\ket{1} \big)$$

So, if the starting register state is $\ket{j_0 j_1 ... j_{n-2}} \otimes \ket{j_{n-1}}$, applying a Hadamard gate to the last (most significant) qubit will result in:

$$\big(I_{n-1} \otimes H \big) \big( \ket{j_0 j_1 ... j_{n-2}} \otimes \ket{j_{n-1}} \big)= \ket{j_0 j_1 ... j_{n-2}} \otimes \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_1}}\ket{1} \big)$$

After this, we can repeat the loop we used in the previous task for qubits $\ket{j_0 j_1 ... j_{n-2}}$ with the last qubit as the target to adjust the remaining phase terms via the controlled $\textrm{Phase}$ gate.

In [ ]:
def binary_fraction_exponent_inplace(j: Qubits) -> None:
    n = j.num_qubits
    j[n - 1].had()
    for ind in range(n - 1):
        j[n - 1].phase((2, 2 ** (n - ind)), cond=j[ind])

## Problem 6. Quantum Fourier transform

Let's use the hint and start by preparing the described state with the qubits reversed:

$$\begin{align*}
& \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_0}} \ket{1} \big) \otimes \\
\otimes & \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_1 j_0}} \ket{1} \big) \otimes ... \\
\otimes & \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_{n-2} ... j_0}} \ket{1} \big) \otimes \\
\otimes & \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_{n-1} j_{n-2} ... j_0}} \ket{1} \big)
\end{align*}$$

You've already found a way to prepare the desired state of the last, most significant qubit in the previous task. The state $\ket{j_0 j_1 ... j_{n-2}} \otimes \tfrac1{\sqrt2} (\ket{0} + e^{2\pi i \cdot \overline{0.j_{n-1} j_{n-2} ... j_0}} \ket{1})$ can be prepared by first applying the Hadamard gate to the last qubit in state $\ket{j_{n-1}}$ and then applying a succession of controlled rotations using each of the qubits $\ket{j_k}$ as controls, so that an extra phase terms from $e^{2\pi i \cdot j_{n-2}/2^2}$ all the way up to $e^{2\pi i \cdot j_0/2^{n}}$ are introduced with each rotation. 

This will prepare the last qubit in the right state. You can see that $j_{n-1}$ doesn't appear in the expression for the first $n-1$ qubits of the target state, so you won't need to use the last qubit in the rest of the code.

To prepare the remaining qubits in the right states, you can work backwards from this state. After you've prepared the last qubit, look at the second-to-last qubit. It needs to be prepared in the state $\tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_{n-2} ... j_0}} \ket{1} \big)$. You can do this using exactly the same procedure you used for the most significant qubit: apply a Hadamard gate to the qubit $\ket{j_{n-2}}$ and then using qubits $\ket{j_{n-3}}$ to $\ket{j_0}$ to apply $n-2$ controlled rotation gates to that qubit.

After this operation, the total system state will be: 

$$\ket{j_0 j_1 ... j_{n-3}} \otimes \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_{n-2} ... j_0}} \ket{1} \big) \otimes 
\otimes \tfrac1{\sqrt2} \big(\ket{0} + e^{2\pi i \cdot \overline{0.j_{n-1} j_{n-2} ... j_0}} \ket{1} \big)$$

The first two steps allow us to see the general pattern: for each qubit $j_k, k = n-1, n-2, ..., 0$:

1. Apply a Hadamard gate to $|j_k\rangle$.
2. Apply $k$ controlled phase gates to the qubit $|j_k\rangle$ as the target, using qubits $|j_{k-1}\rangle$ through $|j_0\rangle$ as the controls, with varying rotation angles.

The effect of these steps will be preparing the state that almost matches the required state, but has the qubit order reversed compared to it. 
You can fix that by using a series of $\textrm{SWAP}$ gates to reverse the order of qubits in the state.

In [ ]:
def quantum_fourier_transform(j: Qubits) -> None:
    n = j.num_qubits
    for ind in range(n, 0, -1):
        binary_fraction_exponent_inplace(j[:ind])
    for ind in range(n // 2):
        j[ind].swap(j[n - ind - 1])

## Problem 7. Prepare equal superposition of all basis vectors

You can think of the required equal superposition of all basis states as the QFT of the state $\ket{0...0}$. Indeed, if $j_0 = j_1 = ... = j_{n-1} = 0$, $j = 0$, and you get the following state:

$$\frac1{\sqrt{2^n}} \sum_{k=0}^{2^n-1} e^{2\pi i \cdot \frac{jk}{2^{n}}} \ket{k} = \frac{1}{\sqrt{2^n}} \sum_{k=0}^{2^n-1} e^{0} \ket{k} = \frac{1}{\sqrt{2^n}} \sum_{k=0}^{2^n-1} \ket{k}$$

This means that you can solve this task by simply applying the QFT to the given qubit register.

In [ ]:
def prepare_equal_superposition(reg: Qubits) -> None:
    from psiqdk.algorithms import QFT
    qft = QFT()
    qft.compute(reg)

## Problem 8. Prepare periodic state

Recall the definition of the QFT: for a basis state $\ket{j}$, its QFT is defined as 

$$\textrm{QFT} \ket{j} = \frac1{\sqrt{2^n}} \sum_{k=0}^{2^n-1} e^{2\pi i \cdot j k/2^{n}} \ket{k}$$

You can see that using $j = freq$ will produce exactly the required state!

To prepare the basis state $\ket{freq}$ on the input register, you can use the `write` method.

In [ ]:
def prepare_periodic_state(reg: Qubits, freq: int) -> None:
    reg.write(freq)
    from psiqdk.algorithms import QFT
    qft = QFT()
    qft.compute(reg)

## Problem 9. Prepare a state with alternating amplitudes

Can we solve this problem using the solution to the previous one with some fixed value of `freq`?

Since the amplitudes of this state have no imaginary parts, it is clear that the value of exponents for an arbitrary amplitude $e^{2\pi i \cdot \frac{freq \cdot k}{2^n}}$ must be either $2\pi i$ to get an amplitude equal $1$ or $\pi i$ for an amplitude of $-1$. 
This means that only $j_{n-1}$ can be $1$: all other bits of $freq$ must be $0$. 
Indeed, you can check that using $freq = \overline{10...0} = 2^{n - 1}$ with the solution of the previous problem yields the required state.

You can simplify preparing this state: instead of using the general encoding of $freq$ in the register, you can apply an $X$ gate to the last (most significant) qubit of the register.

In [ ]:
def prepare_alternating_amplitudes_state(reg: Qubits) -> None:
    prepare_periodic_state(reg, 2 ** (reg.num_qubits - 1))

## Problem 10. Prepare equal superposition of all even basis vectors

You can see that the even superposition of two states from problems 7 and 9 will give the desired result. Indeed,

$$\tfrac1{\sqrt{2^{n-1}}} \big(\ket{0} + \ket{2} + ... + \ket{2^n-2}\big) =$$
$$=\tfrac{1}{\sqrt{2}} \bigg( \tfrac1{\sqrt{2^n}} \big(\ket{0} - \ket{1} + \ket{2} - \ket{3} + ... - \ket{2^n-1}\big) + \tfrac1{\sqrt{2^n}} \big(\ket{0} + \ket{1} + \ket{2} + \ket{3} + ... + \ket{2^n-1}\big)\bigg)$$

Now, you can use the fact that QFT is a linear transformation.
The two periodic states from earlier problems are created by applying the QFT to the states $\ket{0...01}$ and $\ket{0...00}$. 
To prepare an equal superposition of these two states, you can apply a Hadamard gate to the last (most significant) qubit in the register:

$$\tfrac1{\sqrt2} \ket{0...0} \otimes \big(\ket{0} + \ket{1}\big)$$

Then you apply QFT to the register: 

$$\textrm{QFT}\big(\tfrac1{\sqrt2} \ket{0...0} \otimes (\ket{0} + \ket{1}) \big) = \tfrac1{\sqrt2} \big( \textrm{QFT}\ket{0...00} + \textrm{QFT}\ket{0...01}\big)$$

This produces the desired end state. 

In [ ]:
def prepare_equal_superposition_even_states(reg: Qubits) -> None:
    reg[-1].had()
    from psiqdk.algorithms import QFT
    qft = QFT()
    qft.compute(reg)    

## Problem 11. Prepare square-wave signal

If we write the basis states of the goal state in binary instead of integers, we can group them in fours that look as follows:

$$\ket{00...} + \ket{10...} - \ket{01...} - \ket{11...}$$

The states which have $1$ as their second-least significant bit have a $-1$ relative phase, and the states with $0$ in that bit have a $+1$ relative phase. Written as a tensor product of single-qubit states, the target state looks as follows:

$$\ket{+} \otimes \ket{-} \otimes \ket{+}^{\otimes (n-2)}$$

You could try to prepare this state by setting only $j_{n-2}$ to $1$ and the rest of bits of $j$ to $0$ and then applying the QFT, similar to problem 9. This will add the required relative phase of $-1$ on the $\ket{1}$ state of the second-least significant qubit, but will also add a relative phase of $i$ on the $\ket{1}$ state of the least significant qubit:

$$\textrm{QFT}(\ket{0...010}) = \tfrac12 (\ket{0} + i\ket{1}) \otimes (\ket{0} - \ket{1}) \otimes \ket{+}^{\otimes (n-2)}$$

To fix the state of the least significant qubit, you need to cancel out that $i$ relative phase in some form. This can be achieved by using the following state as the input to QFT:

$$\tfrac1{\sqrt2} \big( e^{-i\pi/4} \ket{0...010} + e^{i\pi/4} \ket{0...011} \big)$$

You can write out the result of applying QFT to this state as follows, using the fact that $e^{\pm i\pi/4} = \frac{1 \pm i}{\sqrt2}$:

$$\textrm{QFT} \bigg(\tfrac1{\sqrt2} \big(e^{-i\pi/4} \ket{0...010} + e^{i\pi/4} \ket{0...011}\big) \bigg) = \\

= \tfrac1{\sqrt2} \cdot \tfrac{(1-i)}{\sqrt2} \cdot \tfrac1{\sqrt2} (\ket{0} + i\ket{1}) \otimes \tfrac1{\sqrt2} (\ket{0} - \ket{1}) \otimes \ket{+}^{\otimes (n-2)} + \\

+ \tfrac1{\sqrt2} \cdot \tfrac{(1+i)}{\sqrt2} \cdot \tfrac1{\sqrt2} (\ket{0} - i\ket{1}) \otimes \tfrac1{\sqrt2} (\ket{0} - \ket{1}) \otimes \ket{+}^{\otimes (n-2)} = \\

= \ket{+} \otimes \ket{-} \otimes \ket{+}^{\otimes (n-2)}$$

To prepare the pre-QFT state $\tfrac1{\sqrt2} \big( e^{-i\pi/4} \ket{0...010} + e^{i\pi/4} \ket{0...011} \big) = \ket{0...01} \otimes \tfrac1{\sqrt2} \big( e^{-i\pi/4} \ket{0} + e^{i\pi/4} \ket{1} \big)$, you can use the following sequence of gates:

1. Apply the X gate to second-to-last (second most significant) qubit.
2. Apply the Hadamard gate to the last (most significant) qubit.
3. Apply the Rz gate to the last qubit to apply relative phases $e^{-i\pi/4}$ and $e^{i\pi/4}$ to the basis states $\ket{0}$ and $\ket{1}$, respectively.

In [ ]:
def prepare_square_wave(reg: Qubits) -> None:
    reg[-1].had()
    reg[-2].x()
    reg[-1].rz(90)
    from psiqdk.algorithms import QFT
    qft = QFT()
    qft.compute(reg)

## Problem 12. Get signal frequency

The classical Fourier transform produces a time-domain signal's decomposition in the frequency domain, and the inverse Fourier transform produces a frequency-domain signal's time-domain representation.

The input state is the result of applying QFT to a basis state $\ket{F}$ (see problem 8). This means that the value $F$ can be recovered by applying the inverse QFT to this state and then measuring the qubits to find the resulting basis state.

In [ ]:
def get_signal_frequency(reg: Qubits) -> int:
    from psiqdk.algorithms import QFT
    qft = QFT()
    qft.compute(reg, dagger=True)
    return reg.read()

> Copyright (c) 2026 PsiQuantum